# Lab 1 — Model Evaluation and Data Leakage

**MA553 · Session 2**

Name:

---

A model has been built to predict equipment failure, and it reports an AUC
of almost 1.0 on data held out from training. Your task is to determine why
that score is so high and to produce an estimate that can be defended.

**Part A** introduces two procedures: cross-validation, which rotates which
part of the data is used for evaluation, and pipelines, which keep
preprocessing from seeing the evaluation data.

**Part B** is the audit. Five separate faults inflate the reported score.
Each requires its own correction.

Some cells are marked **Write this yourself**. They give the goal, the
variables to create, and the output to expect. Everything else is written
for you. Work through the cells in order.

# Part A — Cross-validation and pipelines

## The data

Each row is one inspection of one piece of equipment. The same unit is
inspected several times over two years. The target outcome is whether the
unit failed within 30 days of the inspection.

Run this.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("inspections.csv")

print(f"Rows:            {len(df)}")
print(f"Distinct units:  {df.unit_id.nunique()}")
print(f"Failure rate:    {df.failed_within_30d.mean():.1%}")
df.head(3)

Two properties matter later. About one inspection in ten is followed by a
failure, so a rule that always predicts "no failure" is correct most of the
time. And there are far more rows than units, so the same unit appears many
times.

## A1 — One evaluation, start to finish

Run the whole procedure once and name its parts. **Split** the rows into a
training set and a test set, **fit** the model on the training set,
**predict** on the test set, **compare** the predictions with the observed
outcomes, and **report** a metric.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

FEATURES = ["age_months", "hours_since_service", "vibration_mm_s"]
X = df[FEATURES].values
y = df.failed_within_30d.values

# 1. split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
print(f"training rows {len(y_train)}, test rows {len(y_test)}, "
      f"failures in the test set {y_test.sum()}")

# 2. fit
model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)

# 3. predict — each test row receives a score between 0 and 1
scores = model.predict_proba(X_test)[:, 1]
predictions = (scores >= 0.5).astype(int)

In [ ]:
# 4. compare
table = pd.DataFrame(
    confusion_matrix(y_test, predictions),
    index=["no failure observed", "failure observed"],
    columns=["predicted: no failure", "predicted: failure"])
print(table)

# 5. report
print()
print(f"Accuracy:                        {accuracy_score(y_test, predictions):.3f}")
print(f"Accuracy of always predicting no: {1 - y_test.mean():.3f}")
print(f"Failures detected:                {predictions[y_test == 1].sum()} of {y_test.sum()}")

**Answer here.** A missed failure costs about twenty times a false alarm.
Total cost of a threshold:

`cost = 20 * (missed failures) + (false alarms)`

1. Compute this cost for thresholds 0.5, 0.4, 0.3, 0.2 and 0.1. Recompute
   `predictions` at each threshold; the fitted model does not change.
2. Report the cost at each threshold and the one you would recommend.

Note: you chose that threshold by looking at these rows, so they are now
part of model development. A final assessment needs separate test data.

*Your answer:*

Because failures are rare, accuracy is a poor summary. From here on the
models are scored by **ROC AUC**: take one inspection followed by a failure
and one not followed by a failure, and measure how often the model gives the
first the higher score. An AUC of 0.5 is chance-level ranking and 1.0 is
perfect ranking.

In [ ]:
print(f"AUC on this split: {roc_auc_score(y_test, scores):.3f}")

## A2 — A single split is unstable

**Write this yourself.**

A single train/test split gives one AUC. Repeating the split with a
different random assignment gives a different AUC. Measure how much it
varies.

What to do:

1. Create an empty list `split_scores`.
2. Loop over `seed` in `range(20)`. Inside the loop, call
   `train_test_split` exactly as in A1 but pass `random_state=seed`, fit a
   `RandomForestClassifier(n_estimators=200, random_state=0)` on the
   training rows, and append the test-set AUC to `split_scores`.
3. Convert `split_scores` to a numpy array and print its minimum, maximum,
   mean, and standard deviation.

Expected: twenty AUC values between about 0.71 and 0.82, with a mean near
0.77 and a standard deviation near 0.03.

In [ ]:
# Your code here.
split_scores = []

The model and the data are identical across those twenty runs. Only the
assignment of rows to the test set changed.

## A3 — Cross-validation

**Cross-validation** divides the data into equal parts called folds. Each
fold serves once as the evaluation set while the model is fitted on the
remaining folds. Every row is evaluated exactly once, and the fold scores
are averaged.

**Stratified** folds hold the failure rate approximately constant across
folds. With one failure in ten, an unlucky fold could otherwise contain
almost none.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=0),
    X, y, cv=folds, scoring="roc_auc")

print("Score on each fold:", np.round(cv_scores, 3))
print(f"Average: {cv_scores.mean():.3f}")

**Answer here.** Identify the highest- and lowest-scoring folds, then
compare the rows they contain:

1. How many failures does each hold?
2. How many distinct units?
3. Does either difference explain the gap in score?

*Your answer:*

## A4 — Pipelines

Most data requires preprocessing before a model sees it: filling missing
values, putting columns on a common scale, converting text categories into
numbers.

Preprocessing the whole dataset and splitting afterwards means the
preprocessing has already used the evaluation rows. A **pipeline** is a
single object holding the preprocessing steps and the model together. Passed
to cross-validation, its preprocessing is fitted on the training folds only
and then applied to the evaluation fold.

In [ ]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SENSORS = [col for col in df.columns if col.startswith("sensor_")]
NUMERIC = ["age_months", "hours_since_service", "vibration_mm_s", "temperature_c"] + SENSORS
CATEGORICAL = ["site", "inspector"]


def preparation(numeric_columns):
    """Impute missing numbers, standardise them, and encode the text
    columns as numeric indicators."""
    return ColumnTransformer([
        ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
         numeric_columns),
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ])


print("Numeric columns:", len(NUMERIC), "of which sensor channels:", len(SENSORS))

Note the forty sensor channels. In this dataset they carry no signal, but
the code you audit in Part B was written without that knowledge, and it
keeps the twelve columns that appear most related to the outcome.

## A5 — Grouped folds

**Write this yourself.**

Several rows belong to the same unit. If some rows of a unit are used for
fitting and its remaining rows for evaluation, the model can identify the
unit rather than the condition that precedes failure. `GroupKFold` keeps all
rows of a unit within one fold.

What to do:

1. Import `GroupKFold` from `sklearn.model_selection`.
2. Create `grouped = GroupKFold(n_splits=5)`.
3. Loop over `grouped.split(df, y, groups=df.unit_id)`. Each iteration gives
   two arrays of row positions, the fitting rows and the evaluation rows.
4. For each fold, print how many rows are on each side and how many units
   appear on both sides. To count the units on both sides, build a set of
   `df.unit_id` for each side and take the intersection.

Expected: five folds of roughly 340 evaluation rows each, and **zero** units
appearing on both sides in every fold.

Then change `GroupKFold(n_splits=5)` to `StratifiedKFold(n_splits=5,
shuffle=True, random_state=0)`, rerun, and note how the overlap count
changes. `StratifiedKFold.split` does not take a `groups` argument, so call
it as `folds.split(df, y)`.

In [ ]:
# Your code here.

Time order is the other constraint that matters for this kind of data.
Training on later inspections to predict earlier ones is not possible in
operation, so a time-ordered split fits on the past and evaluates on the
future. Session 5 covers it; this lab uses grouping only.

# Part B — The audit

## B1 — Predict first

Before running anything: equipment failure is difficult to predict from
routine inspection readings.

**My prediction.** A defensible AUC for this task is about ______ .

**My reasoning:**

*Your answer:*

## B2 — Run the released code

This is the code as it was handed over.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# ---- the code as it was given to you -------------------------------------
columns = NUMERIC + ["repair_cost_cad"]

prepared = preparation(columns).fit_transform(df[columns + CATEGORICAL])
selected = SelectKBest(f_classif, k=12).fit_transform(prepared, y)

X_tr, X_te, y_tr, y_te = train_test_split(
    selected, y, test_size=0.25, random_state=0, stratify=y)

reported = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                  random_state=0).fit(X_tr, y_tr)
print(f"Reported AUC: {roc_auc_score(y_te, reported.predict_proba(X_te)[:, 1]):.3f}")

**Answer here.** You know from the lecture which five faults are present.
Rank them from the one you expect to cost the most AUC when corrected to the
one you expect to cost the least, and give your reasoning for the first and
last place. Record the ranking now; B8 asks you to compare it with what you
measured.

*Your answer:*

## B3 — Fault one: a feature that could not have been known in advance

Examine `repair_cost_cad` for inspections that were and were not followed by
a failure.

In [ ]:
print(df.groupby("failed_within_30d").repair_cost_cad.describe()[["count", "mean", "max"]])

**Answer here.** Why does including this column make the score meaningless,
and which general rule does it violate?

*Your answer:*

In [ ]:
columns = NUMERIC                      # repair_cost_cad removed
prepared = preparation(columns).fit_transform(df[columns + CATEGORICAL])
selected = SelectKBest(f_classif, k=12).fit_transform(prepared, y)
X_tr, X_te, y_tr, y_te = train_test_split(selected, y, test_size=0.25,
                                          random_state=0, stratify=y)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                               random_state=0).fit(X_tr, y_tr)
print(f"AUC after removing the outcome-derived column: "
      f"{roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]):.3f}")

## B4 — Fault two: the same record on both sides

**Write this yourself.**

Some inspections were entered into the system twice. A duplicated row can be
assigned to the training set while its copy is assigned to the test set.

What to do:

1. Count the exactly duplicated rows with `df.duplicated().sum()` and print
   the count.
2. Create `deduplicated = df.drop_duplicates().reset_index(drop=True)` and
   `y_dedup = deduplicated.failed_within_30d.values`. Later cells use both
   names, so spell them exactly.
3. Rerun the B3 evaluation on `deduplicated` and `y_dedup`: the same
   `preparation(NUMERIC)`, the same `SelectKBest(f_classif, k=12)`, the same
   split arguments, the same model. Print the AUC.

Expected: 126 duplicated rows, 1,586 rows remaining, and an AUC near 0.79.

In [ ]:
# Your code here.

**Answer here.** State in one sentence what the model is being rewarded for
when a row and its copy sit on opposite sides of the split.

*Your answer:*

## B5 — Fault three: preprocessing fitted to all the data

**Write this yourself.**

Two steps were carried out on the whole dataset before splitting: the
imputation and scaling, and the choice of which twelve columns to keep. The
second is the serious one. Forty of these columns are noise, and choosing
the twelve that appear most related to the outcome — using every row,
including the evaluation rows — selects the noise columns that happen to
align with the outcome in those rows.

Compare two procedures on the same rows, folds and model:

- **Before cross-validation.** Imputation, scaling and feature selection are
  fitted once, on every row.
- **Inside each training fold.** The same three steps are fitted separately
  within each fold.

The feature selection step accounts for most of the difference here.
Imputation and scaling contribute very little.

What to do:

1. **Chosen once, outside the folds.** Apply `preparation(NUMERIC)` and
   `SelectKBest(f_classif, k=12)` to all of `deduplicated`, then pass the
   resulting array to `cross_val_score` with
   `RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
   random_state=0)`, `cv=folds`, `scoring="roc_auc"`. Print the mean.
2. **Chosen inside each fold.** Build `honest_pipeline` as a `Pipeline` with
   three steps named `"prepare"`, `"select"` and `"model"`, holding
   `preparation(NUMERIC)`, `SelectKBest(f_classif, k=12)` and the same
   classifier. Pass the pipeline, `deduplicated[NUMERIC + CATEGORICAL]` and
   `y_dedup` to `cross_val_score` with the same `folds`. Print the fold
   scores and their mean.

B6 reuses `honest_pipeline`, so spell that name exactly.

Expected: about 0.78 when the columns are chosen once, about 0.76 when they
are chosen inside each fold.

In [ ]:
# Your code here.

## B6 — Fault four: the same unit on both sides

**Write this yourself.**

Each unit is inspected several times, and units differ from one another in
ways the readings partly capture. Splitting rows at random lets the model
recognise the unit rather than the condition preceding failure. The
objective is performance on units absent from the training data.

What to do:

Call `cross_val_score` with `honest_pipeline`,
`deduplicated[NUMERIC + CATEGORICAL]` and `y_dedup`, but pass
`cv=GroupKFold(n_splits=5)` and `groups=deduplicated.unit_id`. Print the
fold scores and their mean.

Expected: five fold scores between about 0.70 and 0.76, with a mean near
0.74.

In [ ]:
# Your code here.

**Answer here.**

1. The pipeline does not correct this fault. Why not?
2. Compare this change in AUC with the change in B5. Which comparison in B5
   did you use, and why?

Treat the expected scores in these cells as approximate checks, not targets.

*Your answer:*

## B7 — Fault five: selecting the best of many settings

The last fault is not in the data. Below, twenty combinations of
hyperparameters are evaluated and the best result is reported.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# a final test set untouched by development, split by unit
final_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=3)
develop_rows, final_rows = next(final_split.split(deduplicated, y_dedup,
                                                  groups=deduplicated.unit_id))
develop = deduplicated.iloc[develop_rows]
final = deduplicated.iloc[final_rows]

results = []
for leaf in [1, 2, 4, 8, 16]:
    for k in [4, 6, 12, 20]:
        pipe = Pipeline([
            ("prepare", preparation(NUMERIC)),
            ("select", SelectKBest(f_classif, k=k)),
            ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf,
                                             random_state=0)),
        ])
        s = cross_val_score(pipe, develop[NUMERIC + CATEGORICAL],
                            develop.failed_within_30d, cv=GroupKFold(4),
                            groups=develop.unit_id, scoring="roc_auc").mean()
        results.append((leaf, k, s))

results.sort(key=lambda r: -r[2])
print("Best setting found during development:", results[0])
print("Worst:", results[-1])

In [ ]:
leaf, k, development_score = results[0]
best = Pipeline([
    ("prepare", preparation(NUMERIC)),
    ("select", SelectKBest(f_classif, k=k)),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf,
                                     random_state=0)),
]).fit(develop[NUMERIC + CATEGORICAL], develop.failed_within_30d)

final_score = roc_auc_score(final.failed_within_30d,
                            best.predict_proba(final[NUMERIC + CATEGORICAL])[:, 1])

print(f"Score that selected this setting:  {development_score:.3f}")
print(f"Score on the untouched final set:  {final_score:.3f}")

**Answer here.** Twenty settings were evaluated and the best was reported.
Explain why that best score is not a fair estimate of how the selected model
performs on new data.

Two cautions:

- **This run.** The two scores come from different rows, so their difference
  also contains ordinary sampling variation. On a single split the test score
  can fall on either side of the development score. The optimism from
  selecting a winner appears as an average over repetitions. It grows with
  the number of settings evaluated and shrinks as the data grow.
- **Scope of this test set.** Its units are excluded from B7's search, but
  earlier exercises in this notebook used those same rows. B7 demonstrates
  the correct workflow. It is not an independent test of the whole audit.
  Reserving units at the very start would fix this, and is a small change to
  the code.

*Your answer:*

The cell below repeats the procedure across several splits. It uses fewer
settings and fewer trees to keep the runtime short, so it shows the same
effect on a smaller scale rather than repeating the search above. It takes a
few minutes, so run it outside class, then answer: what is the average gap,
and in which direction does it point?

*Your answer:*

In [ ]:
# Slow. Run this outside class.
from sklearn.model_selection import GroupShuffleSplit

gaps = []
for run in range(6):
    rows_dev, rows_final = next(GroupShuffleSplit(
        n_splits=1, test_size=0.25, random_state=run).split(
            deduplicated, y_dedup, groups=deduplicated.unit_id))
    dev, fin = deduplicated.iloc[rows_dev], deduplicated.iloc[rows_final]

    best_score, best_setting = -1, None
    for leaf in [1, 4, 16]:
        for k in [4, 12]:
            candidate = Pipeline([
                ("prepare", preparation(NUMERIC)),
                ("select", SelectKBest(f_classif, k=k)),
                ("model", RandomForestClassifier(n_estimators=120,
                                                 min_samples_leaf=leaf,
                                                 random_state=0))])
            sc = cross_val_score(candidate, dev[NUMERIC + CATEGORICAL],
                                 dev.failed_within_30d, cv=GroupKFold(4),
                                 groups=dev.unit_id, scoring="roc_auc").mean()
            if sc > best_score:
                best_score, best_setting = sc, (leaf, k)

    leaf, k = best_setting
    chosen = Pipeline([
        ("prepare", preparation(NUMERIC)),
        ("select", SelectKBest(f_classif, k=k)),
        ("model", RandomForestClassifier(n_estimators=120, min_samples_leaf=leaf,
                                         random_state=0))
    ]).fit(dev[NUMERIC + CATEGORICAL], dev.failed_within_30d)

    on_final = roc_auc_score(fin.failed_within_30d,
                             chosen.predict_proba(fin[NUMERIC + CATEGORICAL])[:, 1])
    gaps.append(best_score - on_final)
    print(f"run {run}: development {best_score:.3f}  untouched {on_final:.3f}  "
          f"gap {best_score - on_final:+.3f}")

print("Average gap over", len(gaps), "runs:", round(float(np.mean(gaps)), 3))

## B8 — The record of corrections

Fill in one row per correction, in the order you applied them. Record how
each AUC was measured: a single split, five-fold cross-validation, grouped
folds, or a test set.

| Cell | Correction applied | How AUC was measured | AUC before | AUC after |
|---|---|---|---|---|
| B3 | | | | |
| B4 | | | | |
| B5 | | | | |
| B6 | | | | |
| B7 | | | | |

Two limits on reading this table:

- The corrections are cumulative. Each AUC is measured with all earlier
  corrections in place, so a change is not that fault's cost on its own.
- Some steps also change how performance is measured. A difference between
  two rows measured in different ways is not the effect of the correction
  alone. B5 is the exception: its outside-versus-inside comparison holds the
  rows, folds and model fixed.

**Answer here.** Compare what you measured with the ranking you recorded in
B2. Where were you wrong, and what would have told you?

*Your answer:*

## B9 — The judgment

**1. Report both estimates, labelled separately.**

- From B6: the mean grouped cross-validation AUC and the five fold scores.
- From B7: the best development AUC and the final test AUC.

State why grouping by unit suits this prediction task.

*Your answer:*

**2. Compare** those estimates with the 1.0 you started from, and with the
prediction you recorded in B1.

*Your answer:*

**3. State one conclusion and one limitation.** One sentence each. The
limitation should take account of what B7's test set does and does not
cover.

*Your answer:*

## Before you submit

- Restart the kernel and run every cell from top to bottom.
- Check that every **Write this yourself** cell runs and produces the
  expected output, and that every **Answer here** cell and the B8 table are
  filled in.
- If you used an AI assistant, note what it produced and what you changed.
- Submit through Moodle.